# Kaggle Studio
Ative Internet e GPU T4 ×2. Execute preparação, depois runtime.
Interromper runtime encerra servidores. Túnel rápido serve para testes sem SSE; use túnel nomeado para agentes.


In [ ]:
import os, shutil, subprocess, sys, venv
from pathlib import Path

RUNTIME_VENV = Path("/kaggle/working/.kaggle-runtime-venv")
RUNTIME_PYTHON = RUNTIME_VENV / "bin" / "python"
CLEAN_ENV = os.environ.copy()
CLEAN_ENV.pop("PYTHONPATH", None)
CLEAN_ENV.pop("PYTHONHOME", None)
CLEAN_ENV["PYTHONNOUSERSITE"] = "1"

def runtime_python_ok():
    return RUNTIME_PYTHON.exists() and subprocess.run(
        [str(RUNTIME_PYTHON), "-c", "import sys; print(sys.prefix)"],
        stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL,
        env=CLEAN_ENV,
    ).returncode == 0

if not runtime_python_ok():
    print("Criando venv isolado sem ensurepip:", RUNTIME_VENV)
    if RUNTIME_VENV.exists():
        shutil.rmtree(RUNTIME_VENV)
    venv.EnvBuilder(with_pip=False, clear=False, symlinks=False).create(RUNTIME_VENV)

# --python instala NO venv alvo. Ambiente Python global não é alterado.
install = [
    sys.executable, "-m", "pip", "--python", str(RUNTIME_PYTHON),
    "install", "--no-cache-dir", "-q",
    "--upgrade-strategy", "only-if-needed",
    "huggingface_hub", "hf_xet", "fastapi", "uvicorn", "httpx", "requests", "tqdm", "uvloop", "httptools",
]
result = subprocess.run(install, env=CLEAN_ENV)
if result.returncode:
    # Compatibilidade com pip antigo: bootstrap direto dentro do venv.
    get_pip = Path("/kaggle/working/get-pip.py")
    subprocess.check_call([
        sys.executable, "-c",
        "import urllib.request; urllib.request.urlretrieve('https://bootstrap.pypa.io/get-pip.py', r'%s')" % get_pip,
    ], env=CLEAN_ENV)
    subprocess.check_call([str(RUNTIME_PYTHON), str(get_pip), "--no-cache-dir", "-q"], env=CLEAN_ENV)
    get_pip.unlink(missing_ok=True)
    subprocess.check_call([str(RUNTIME_PYTHON), "-m", "pip", *install[5:]], env=CLEAN_ENV)

subprocess.check_call([str(RUNTIME_PYTHON), "-c", "import fastapi,httpx,huggingface_hub,uvicorn,requests"], env=CLEAN_ENV)
print("Venv runtime pronto:", RUNTIME_PYTHON)


In [ ]:
# Executa runtime no venv, sem injetar pacotes no kernel Kaggle.
import base64, os, subprocess
from pathlib import Path
runtime_file = Path("/kaggle/working/kaggle_studio_runtime.py")
runtime_file.write_bytes(base64.b64decode('...'))
runtime_file.chmod(0o600)
runtime_env = os.environ.copy()
runtime_env.pop("PYTHONPATH", None)
runtime_env.pop("PYTHONHOME", None)
runtime_env["PYTHONNOUSERSITE"] = "1"
runtime_env["KAGGLE_TUNNEL_MODE"] = 'quick'
runtime_env["KAGGLE_TUNNEL_URL"] = ''
if runtime_env["KAGGLE_TUNNEL_MODE"] == "named":
    from getpass import getpass
    if not runtime_env["KAGGLE_TUNNEL_URL"].startswith("https://"):
        raise ValueError("Configure URL HTTPS do túnel nomeado no Studio.")
    runtime_env["TUNNEL_TOKEN"] = os.environ.get("TUNNEL_TOKEN") or getpass("Token do túnel Cloudflare (oculto): ")
runtime_python = Path("/kaggle/working/.kaggle-runtime-venv/bin/python")
if not runtime_python.exists():
    raise RuntimeError("Execute primeiro célula 1: preparação.")
process = subprocess.Popen([str(runtime_python), "-u", str(runtime_file)], env=runtime_env, start_new_session=True)
try:
    if process.wait():
        raise RuntimeError("Runtime falhou. Veja erro e logs acima.")
except KeyboardInterrupt:
    print("Runtime, gateway e túnel encerrados.")
finally:
    import signal
    try:
        os.killpg(process.pid, signal.SIGTERM)
        process.wait(timeout=15)
    except ProcessLookupError:
        pass
    except subprocess.TimeoutExpired:
        os.killpg(process.pid, signal.SIGKILL)
        process.wait()
